# Module 7.1: 推理优化 (Inference Optimization)

## 1. 概述

### 学习目标

- 理解推理优化的核心指标（延迟、吞吐量、内存）
- 掌握量化技术（动态量化、静态量化）
- 实现模型剪枝（非结构化、结构化）
- 理解知识蒸馏的原理和实现
- 掌握ONNX模型导出和图优化
- 了解硬件加速方案（TensorRT、OpenVINO）
- 构建综合推理优化流水线

### 核心概念速览

**剪枝 (Pruning)** 是通过移除模型中不重要的权重或结构来压缩模型的技术。非结构化剪枝将单个权重置零，结构化剪枝移除整个神经元/通道/注意力头。在本章场景中，它用于在可接受的精度损失下减少模型参数量和计算量。

**知识蒸馏 (Knowledge Distillation)** 是将大型教师模型的知识迁移到小型学生模型的模型压缩技术，通过让学生模型学习教师模型的软标签（soft label）输出分布来实现。在本章场景中，它用于训练出体积小但性能接近大模型的轻量级部署模型。

**ONNX (Open Neural Network Exchange)** 是一种开放的神经网络模型交换格式，定义了通用的算子集和文件格式，使模型可以在不同深度学习框架之间互相转换。在本章场景中，它用于将 PyTorch 模型导出为通用格式，以便进行图优化和跨平台部署。

**TensorRT** 是 NVIDIA 开发的高性能深度学习推理优化引擎，通过算子融合、精度校准、内核自动调优等技术最大化 GPU 推理性能。在本章场景中，它用于在 NVIDIA GPU 上实现最优推理延迟和吞吐量。

**OpenVINO (Open Visual Inference and Neural Network Optimization)** 是 Intel 开发的推理优化工具套件，针对 Intel CPU/GPU/VPU 等硬件进行专门优化。在本章场景中，它用于在 Intel 硬件上高效部署模型。


### 🏢 业务场景

本章技术将应用于 `电商客服智能助理` 场景：
- 客服高峰期响应延迟超 SLA？→ 模型量化与推理加速技术
- 压缩后回复质量下降怎么控制？→ 量化精度与效果的平衡策略

### 知识地图

```
推理优化
├── 基础分析 ─── 延迟/吞吐量/内存 / Roofline模型
├── 量化 ─── 动态量化 / 静态量化 / 对称vs非对称
├── 剪枝 ─── 非结构化 / 结构化 / 重要性评估
├── 知识蒸馏 ─── 教师-学生 / 温度参数 / 蒸馏损失
├── 模型导出 ─── ONNX / 图优化 / 算子融合
├── 硬件加速 ─── TensorRT / OpenVINO / CoreML
└── 综合优化 ─── 流水线组合 / 决策树
```

### 为什么需要推理优化？

训练好的模型在部署时面临严格约束：

1. **延迟要求**：实时应用需要毫秒级响应
2. **成本压力**：GPU推理成本高昂
3. **设备限制**：边缘设备内存和算力有限
4. **规模挑战**：每天处理数百万请求

本notebook将系统介绍各种推理优化技术。

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)
print("✓ Libraries imported")

## 2. 推理优化基础

### 2.1 优化目标

| 指标 | 定义 | 重要场景 |
|------|------|---------|
| **延迟 (Latency)** | 单次推理时间 | 实时对话、搜索 |
| **吞吐量 (Throughput)** | 每秒处理请求数 | 批处理、离线分析 |
| **内存占用 (Memory)** | 模型+推理内存 | 边缘设备、移动端 |
| **计算成本 (Cost)** | GPU/CPU使用费用 | 大规模部署 |

### 2.2 优化层次

```
Level 1: 算法优化
├── 更高效的模型架构
├── 注意力优化（Flash Attention）
└── 推测解码（Speculative Decoding）

Level 2: 模型压缩
├── 量化（Quantization）
├── 剪枝（Pruning）
└── 知识蒸馏（Distillation）

Level 3: 系统优化
├── 模型导出（ONNX）
├── 图优化（算子融合）
└── 批处理策略

Level 4: 硬件加速
├── TensorRT（GPU）
├── OpenVINO（CPU）
└── 专用加速器（TPU/NPU）
```

### 2.3 推理 vs 训练

| 特性 | 训练 | 推理 |
|------|------|------|
| 精度要求 | FP32/BF16 | INT8/INT4可接受 |
| 批大小 | 大（提高利用率） | 小（降低延迟） |
| 内存 | 需要梯度+优化器 | 只需模型参数 |
| 计算模式 | 前向+反向 | 仅前向 |
| 优化重点 | 吞吐量 | 延迟 |

### 2.4 Roofline模型

推理性能受限于两个因素：
- **计算瓶颈**：算力不足（FLOPS）
- **内存瓶颈**：带宽不足（GB/s）

$$\text{Arithmetic Intensity} = \frac{\text{FLOPs}}{\text{Bytes Accessed}}$$

- 高AI → 计算受限（大矩阵乘法）
- 低AI → 内存受限（小batch推理）

In [ ]:
# 🔬 Micro Practice: Baseline Inference Profiling
# Goal: Measure baseline inference performance
# Expected outcome: Understand key inference metrics

class InferenceProfiler:
    """
    Profile model inference performance
    Measures latency, throughput, and memory usage
    """
    
    def __init__(self, model, input_shape):
        self.model = model
        self.input_shape = input_shape
        self.model.eval()
    
    def measure_latency(self, n_runs=100, warmup=10):
        """Measure inference latency (ms)"""
        x = torch.randn(*self.input_shape)
        
        # Warmup
        with torch.no_grad():
            for _ in range(warmup):
                _ = self.model(x)
        
        # Measure
        latencies = []
        with torch.no_grad():
            for _ in range(n_runs):
                start = time.time()
                _ = self.model(x)
                latencies.append((time.time() - start) * 1000)  # ms
        
        return {
            'mean': np.mean(latencies),
            'std': np.std(latencies),
            'p50': np.percentile(latencies, 50),
            'p95': np.percentile(latencies, 95),
            'p99': np.percentile(latencies, 99),
            'min': np.min(latencies),
            'max': np.max(latencies),
        }
    
    def measure_throughput(self, batch_sizes=[1, 4, 8, 16, 32, 64], n_runs=50):
        """Measure throughput at different batch sizes"""
        results = []
        
        for bs in batch_sizes:
            shape = (bs,) + self.input_shape[1:]
            x = torch.randn(*shape)
            
            with torch.no_grad():
                # Warmup
                for _ in range(5):
                    _ = self.model(x)
                
                # Measure
                start = time.time()
                for _ in range(n_runs):
                    _ = self.model(x)
                elapsed = time.time() - start
            
            throughput = bs * n_runs / elapsed
            latency = elapsed / n_runs * 1000
            
            results.append({
                'batch_size': bs,
                'throughput': throughput,
                'latency_ms': latency,
                'latency_per_sample': latency / bs,
            })
        
        return results
    
    def profile_model_size(self):
        """Profile model size and parameter count"""
        total_params = sum(p.numel() for p in self.model.parameters())
        trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        
        # Memory by dtype
        memory = sum(p.numel() * p.element_size() for p in self.model.parameters())
        
        return {
            'total_params': total_params,
            'trainable_params': trainable_params,
            'memory_mb': memory / 1e6,
            'memory_per_param': memory / total_params if total_params > 0 else 0,
        }

# Create a sample model
print("=== Inference Profiling ===\n")

model = nn.Sequential(
    nn.Linear(256, 512),
    nn.ReLU(),
    nn.Linear(512, 512),
    nn.ReLU(),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

profiler = InferenceProfiler(model, input_shape=(1, 256))

# 1. Model size
print("--- Model Size ---")
size_info = profiler.profile_model_size()
print(f"  Total parameters:    {size_info['total_params']:,}")
print(f"  Memory:              {size_info['memory_mb']:.2f} MB")
print(f"  Bytes per parameter: {size_info['memory_per_param']:.0f}")

# 2. Latency
print("\n--- Latency (batch_size=1) ---")
latency = profiler.measure_latency(n_runs=200)
print(f"  Mean:  {latency['mean']:.3f} ms")
print(f"  Std:   {latency['std']:.3f} ms")
print(f"  P50:   {latency['p50']:.3f} ms")
print(f"  P95:   {latency['p95']:.3f} ms")
print(f"  P99:   {latency['p99']:.3f} ms")

# 3. Throughput
print("\n--- Throughput ---")
throughput_results = profiler.measure_throughput()
print(f"  {'Batch Size':<12} {'Throughput':<18} {'Latency':<14} {'Per Sample':<12}")
print("  " + "-" * 56)
for r in throughput_results:
    print(f"  {r['batch_size']:<12} {r['throughput']:<18.0f} samples/s "
          f"{r['latency_ms']:<14.3f} ms {r['latency_per_sample']:<12.3f} ms")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Latency distribution
latencies_raw = []
x = torch.randn(1, 256)
model.eval()
with torch.no_grad():
    for _ in range(500):
        start = time.time()
        _ = model(x)
        latencies_raw.append((time.time() - start) * 1000)

axes[0].hist(latencies_raw, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(x=np.mean(latencies_raw), color='red', linestyle='--', label=f'Mean: {np.mean(latencies_raw):.3f}ms')
axes[0].axvline(x=np.percentile(latencies_raw, 95), color='orange', linestyle='--', label=f'P95: {np.percentile(latencies_raw, 95):.3f}ms')
axes[0].set_xlabel('Latency (ms)')
axes[0].set_ylabel('Count')
axes[0].set_title('Latency Distribution', fontsize=12, weight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Throughput vs Batch Size
batch_sizes = [r['batch_size'] for r in throughput_results]
throughputs = [r['throughput'] for r in throughput_results]

axes[1].plot(batch_sizes, throughputs, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Batch Size')
axes[1].set_ylabel('Throughput (samples/s)')
axes[1].set_title('Throughput vs Batch Size', fontsize=12, weight='bold')
axes[1].grid(True, alpha=0.3)

# Per-sample latency vs Batch Size
per_sample = [r['latency_per_sample'] for r in throughput_results]

axes[2].plot(batch_sizes, per_sample, 'ro-', linewidth=2, markersize=8)
axes[2].set_xlabel('Batch Size')
axes[2].set_ylabel('Per-Sample Latency (ms)')
axes[2].set_title('Per-Sample Latency vs Batch Size', fontsize=12, weight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n核心观察：")
print("- 延迟分布通常有长尾（P99 >> P50）")
print("- 增大batch size可以提高吞吐量（GPU并行）")
print("- 但batch size过大会增加单次延迟")
print("- 需要在延迟和吞吐量之间权衡")
print("\n✓ 基线推理分析完成！")

## 3. 量化 (Quantization)

### 3.1 量化基础

**量化**：将模型权重和激活值从高精度（FP32）转换为低精度（INT8/INT4）。

**量化公式**：

$$x_q = \text{round}\left(\frac{x}{\text{scale}}\right) + \text{zero\_point}$$

$$\text{scale} = \frac{x_{\max} - x_{\min}}{2^b - 1}$$

**反量化**：

$$\hat{x} = (x_q - \text{zero\_point}) \times \text{scale}$$

### 3.2 量化类型

| 类型 | 权重 | 激活值 | 校准数据 | 适用场景 |
|------|------|--------|---------|---------|
| **动态量化** | 静态量化 | 动态量化 | 不需要 | 线性层密集模型 |
| **静态量化** | 静态量化 | 静态量化 | 需要 | 追求最佳性能 |
| **QAT** | 训练中量化 | 训练中量化 | 训练数据 | 精度敏感场景 |

### 3.3 对称 vs 非对称量化

**对称量化**：
- 范围：$[-\max|x|, \max|x|]$
- zero_point = 0
- 简单高效，适合权重

**非对称量化**：
- 范围：$[x_{\min}, x_{\max}]$
- zero_point ≠ 0
- 更精确，适合ReLU后的激活值（非负）

### 3.4 精度对比

| 精度 | 位宽 | 模型大小 | 推理速度 | 精度损失 |
|------|------|---------|---------|---------|
| FP32 | 32 bit | 1x | 1x | 0% |
| FP16 | 16 bit | 0.5x | 1.5-2x | ~0% |
| INT8 | 8 bit | 0.25x | 2-4x | <1% |
| INT4 | 4 bit | 0.125x | 3-6x | 2-5% |

In [ ]:
# 🔬 Micro Practice: Quantization for Inference
# Goal: Implement dynamic and static quantization
# Expected outcome: Understand quantization trade-offs for deployment

class Quantizer:
    """
    Quantization toolkit for model compression
    Supports dynamic and static quantization
    """
    
    @staticmethod
    def quantize_tensor(x, num_bits=8, symmetric=True):
        """
        Quantize a floating-point tensor to fixed-point
        
        Args:
            x: Input tensor (float)
            num_bits: Target bit width
            symmetric: Use symmetric or asymmetric quantization
        """
        if symmetric:
            # Symmetric: range [-max_abs, max_abs]
            max_abs = x.abs().max()
            qmin = -(2**(num_bits-1))
            qmax = 2**(num_bits-1) - 1
            scale = max_abs / qmax if qmax > 0 else 1.0
            zero_point = 0
        else:
            # Asymmetric: range [min, max]
            x_min, x_max = x.min(), x.max()
            qmin = 0
            qmax = 2**num_bits - 1
            scale = (x_max - x_min) / (qmax - qmin) if qmax > qmin else 1.0
            zero_point = qmin - torch.round(x_min / scale)
        
        # Quantize
        x_q = torch.clamp(torch.round(x / scale + zero_point), qmin, qmax)
        
        # Dequantize
        x_dq = (x_q - zero_point) * scale
        
        return x_dq, scale, zero_point
    
    @staticmethod
    def dynamic_quantize_model(model, num_bits=8):
        """
        Dynamic quantization: quantize weights statically, activations dynamically
        - Weights: quantized once after training
        - Activations: quantized on-the-fly during inference
        """
        import copy
        q_model = copy.deepcopy(model)
        
        with torch.no_grad():
            for name, param in q_model.named_parameters():
                if 'weight' in name:
                    param.data, _, _ = Quantizer.quantize_tensor(param.data, num_bits)
        
        return q_model
    
    @staticmethod
    def static_quantize_model(model, calibration_data, num_bits=8):
        """
        Static quantization: quantize both weights and activations
        Uses calibration data to determine activation ranges
        
        Steps:
        1. Run calibration data through model
        2. Collect activation statistics (min, max)
        3. Compute quantization parameters
        4. Apply quantization
        """
        import copy
        q_model = copy.deepcopy(model)
        
        # Step 1: Collect activation statistics
        activation_ranges = {}
        hooks = []
        
        def make_hook(name):
            def hook_fn(module, input, output):
                if name not in activation_ranges:
                    activation_ranges[name] = {'min': float('inf'), 'max': float('-inf')}
                activation_ranges[name]['min'] = min(activation_ranges[name]['min'], output.min().item())
                activation_ranges[name]['max'] = max(activation_ranges[name]['max'], output.max().item())
            return hook_fn
        
        for name, module in q_model.named_modules():
            if isinstance(module, nn.Linear):
                hooks.append(module.register_forward_hook(make_hook(name)))
        
        # Run calibration
        q_model.eval()
        with torch.no_grad():
            for data in calibration_data:
                _ = q_model(data)
        
        # Remove hooks
        for h in hooks:
            h.remove()
        
        # Step 2: Quantize weights
        with torch.no_grad():
            for name, param in q_model.named_parameters():
                if 'weight' in name:
                    param.data, _, _ = Quantizer.quantize_tensor(param.data, num_bits)
        
        return q_model, activation_ranges

# Demo quantization
print("=== Quantization for Inference ===\n")

# Create and train a model
input_dim, hidden_dim, output_dim = 64, 256, 10

model = nn.Sequential(
    nn.Linear(input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim, output_dim)
)

X_train = torch.randn(1000, input_dim)
y_train = torch.randint(0, output_dim, (1000,))
X_test = torch.randn(200, input_dim)
y_test = torch.randint(0, output_dim, (200,))

# Train
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(100):
    optimizer.zero_grad()
    loss = nn.functional.cross_entropy(model(X_train), y_train)
    loss.backward()
    optimizer.step()

# Evaluate original
model.eval()
with torch.no_grad():
    original_acc = (model(X_test).argmax(dim=-1) == y_test).float().mean().item()

print(f"Original model accuracy: {original_acc*100:.1f}%")
print()

# Dynamic quantization at different bit widths
print("=== Dynamic Quantization ===\n")
print(f"{'Bits':<8} {'Accuracy':<12} {'Acc Drop':<12} {'Size Ratio':<12}")
print("-" * 44)

quant_results = []
for bits in [2, 4, 8, 16]:
    q_model = Quantizer.dynamic_quantize_model(model, num_bits=bits)
    
    with torch.no_grad():
        q_acc = (q_model(X_test).argmax(dim=-1) == y_test).float().mean().item()
    
    acc_drop = (original_acc - q_acc) * 100
    size_ratio = bits / 32 * 100
    
    quant_results.append((bits, q_acc, acc_drop, size_ratio))
    print(f"{bits:<8} {q_acc*100:<12.1f}% {acc_drop:<12.1f}% {size_ratio:<12.1f}%")

# Static quantization
print("\n=== Static Quantization (INT8) ===\n")

calibration_data = [X_train[i:i+32] for i in range(0, 256, 32)]
sq_model, act_ranges = Quantizer.static_quantize_model(model, calibration_data, num_bits=8)

with torch.no_grad():
    sq_acc = (sq_model(X_test).argmax(dim=-1) == y_test).float().mean().item()

print(f"Static quantized accuracy: {sq_acc*100:.1f}%")
print(f"Accuracy drop: {(original_acc - sq_acc)*100:.1f}%")
print(f"\nActivation ranges (from calibration):")
for name, ranges in act_ranges.items():
    print(f"  {name}: [{ranges['min']:.3f}, {ranges['max']:.3f}]")

# Benchmark speed
print("\n=== Speed Benchmark ===\n")

x_bench = torch.randn(1, input_dim)
n_runs = 500

# Original
start = time.time()
with torch.no_grad():
    for _ in range(n_runs):
        _ = model(x_bench)
original_time = (time.time() - start) / n_runs * 1000

# INT8 dynamic
q8_model = Quantizer.dynamic_quantize_model(model, num_bits=8)
start = time.time()
with torch.no_grad():
    for _ in range(n_runs):
        _ = q8_model(x_bench)
q8_time = (time.time() - start) / n_runs * 1000

print(f"Original (FP32): {original_time:.4f} ms")
print(f"Quantized (INT8): {q8_time:.4f} ms")
print(f"Note: Real INT8 speedup requires hardware support (2-4x on CPU)")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Accuracy vs Bits
bits_list = [r[0] for r in quant_results]
accs = [r[1]*100 for r in quant_results]

axes[0].plot(bits_list, accs, 'bo-', linewidth=2, markersize=10)
axes[0].axhline(y=original_acc*100, color='red', linestyle='--', label='Original (FP32)')
axes[0].set_xlabel('Bit Width', fontsize=12)
axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_title('Accuracy vs Quantization Bits', fontsize=12, weight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Model size comparison
sizes = [r[3] for r in quant_results]
colors = ['#F44336', '#FF9800', '#4CAF50', '#2196F3']
axes[1].bar([str(b) + '-bit' for b in bits_list], sizes, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Size (% of FP32)')
axes[1].set_title('Model Size Reduction', fontsize=12, weight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# Quantization error distribution
original_weights = torch.cat([p.flatten() for p in model.parameters()]).detach()
q8_weights = torch.cat([p.flatten() for p in q8_model.parameters()]).detach()
quant_error = (original_weights - q8_weights).numpy()

axes[2].hist(quant_error, bins=100, color='steelblue', alpha=0.7, edgecolor='black')
axes[2].set_xlabel('Quantization Error')
axes[2].set_ylabel('Count')
axes[2].set_title('INT8 Quantization Error Distribution', fontsize=12, weight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n量化核心要点：")
print("- INT8量化通常精度损失<1%，模型大小减少4x")
print("- 动态量化简单易用，适合线性层密集的模型")
print("- 静态量化需要校准数据，但效果更好")
print("- INT4量化精度损失较大，适合对精度不敏感的场景")
print("\n✓ 量化技术完成！")

## 4. 模型剪枝 (Pruning)

### 4.1 核心思想

**剪枝**：移除模型中不重要的权重或神经元，减少模型大小和计算量。

**理论基础**：彩票假说（Lottery Ticket Hypothesis）
> 随机初始化的密集网络中包含一个稀疏子网络（"中奖彩票"），该子网络单独训练可以达到与原网络相当的性能。

### 4.2 剪枝类型

**非结构化剪枝（Unstructured Pruning）**：
- 移除单个权重（设为0）
- 产生稀疏矩阵
- 灵活性高，但需要稀疏硬件支持
- 例：移除绝对值最小的50%权重

**结构化剪枝（Structured Pruning）**：
- 移除整个神经元、通道或注意力头
- 直接减少模型维度
- 不需要特殊硬件支持
- 例：移除L1范数最小的30%神经元

### 4.3 重要性评估

| 方法 | 公式 | 优点 | 缺点 |
|------|------|------|------|
| **幅度剪枝** | $\|w\|$ | 简单高效 | 忽略梯度信息 |
| **梯度剪枝** | $\|w \cdot \nabla w\|$ | 考虑损失影响 | 计算开销大 |
| **Fisher剪枝** | $w^2 \cdot (\nabla w)^2$ | 理论基础好 | 需要二阶信息 |
| **随机剪枝** | 随机选择 | 基线对比 | 效果差 |

### 4.4 剪枝流程

```
训练完整模型 → 评估重要性 → 剪枝 → 微调恢复 → 评估
                                    ↑              ↓
                                    └──── 迭代 ────┘
```

In [ ]:
# 🔬 Micro Practice: Model Pruning
# Goal: Implement magnitude-based and structured pruning
# Expected outcome: Understand how to remove unimportant weights

class MagnitudePruner:
    """
    Magnitude-based pruning: remove weights with smallest absolute values
    Assumption: small weights contribute less to the output
    """
    
    @staticmethod
    def unstructured_prune(model, sparsity=0.5):
        """
        Unstructured pruning: remove individual weights
        Creates sparse weight matrices
        """
        masks = {}
        
        for name, param in model.named_parameters():
            if 'weight' in name and param.dim() >= 2:
                # Compute threshold
                threshold = torch.quantile(param.abs().float(), sparsity)
                
                # Create mask (1 = keep, 0 = prune)
                mask = (param.abs() > threshold).float()
                masks[name] = mask
                
                # Apply mask
                param.data *= mask
        
        return masks
    
    @staticmethod
    def structured_prune(model, sparsity=0.3):
        """
        Structured pruning: remove entire neurons/channels
        Based on L1-norm of each output neuron's weights
        """
        prune_info = {}
        
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                # Compute importance of each output neuron (L1 norm)
                importance = module.weight.data.abs().sum(dim=1)
                
                # Find neurons to prune
                n_prune = int(len(importance) * sparsity)
                _, indices = importance.sort()
                prune_indices = indices[:n_prune]
                keep_indices = indices[n_prune:]
                
                # Zero out pruned neurons
                module.weight.data[prune_indices] = 0
                if module.bias is not None:
                    module.bias.data[prune_indices] = 0
                
                prune_info[name] = {
                    'total_neurons': len(importance),
                    'pruned_neurons': n_prune,
                    'kept_neurons': len(importance) - n_prune,
                }
        
        return prune_info

def compute_sparsity(model):
    """Compute overall model sparsity"""
    total_params = 0
    zero_params = 0
    
    for param in model.parameters():
        total_params += param.numel()
        zero_params += (param == 0).sum().item()
    
    return zero_params / total_params * 100

def evaluate_model(model, X, y):
    """Evaluate model accuracy"""
    model.eval()
    with torch.no_grad():
        output = model(X)
        acc = (output.argmax(dim=-1) == y).float().mean().item()
    return acc

# Setup
print("=== Model Pruning ===\n")

input_dim, hidden_dim, output_dim = 64, 256, 10

# Create and train a model
model = nn.Sequential(
    nn.Linear(input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim, output_dim)
)

X_train = torch.randn(1000, input_dim)
y_train = torch.randint(0, output_dim, (1000,))
X_test = torch.randn(200, input_dim)
y_test = torch.randint(0, output_dim, (200,))

# Train
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(100):
    optimizer.zero_grad()
    loss = nn.functional.cross_entropy(model(X_train), y_train)
    loss.backward()
    optimizer.step()

original_acc = evaluate_model(model, X_test, y_test)
print(f"Original model accuracy: {original_acc*100:.1f}%")
print(f"Original sparsity: {compute_sparsity(model):.1f}%")
print()

# Test different sparsity levels
print("=== Unstructured Pruning at Different Sparsity Levels ===\n")
print(f"{'Sparsity':<12} {'Actual Sparsity':<18} {'Accuracy':<12} {'Acc Drop':<12}")
print("-" * 54)

import copy
sparsity_levels = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95]
results_unstructured = []

for sparsity in sparsity_levels:
    pruned_model = copy.deepcopy(model)
    MagnitudePruner.unstructured_prune(pruned_model, sparsity)
    
    actual_sparsity = compute_sparsity(pruned_model)
    acc = evaluate_model(pruned_model, X_test, y_test)
    acc_drop = (original_acc - acc) * 100
    
    results_unstructured.append((sparsity, actual_sparsity, acc, acc_drop))
    print(f"{sparsity*100:<12.0f}% {actual_sparsity:<18.1f}% {acc*100:<12.1f}% {acc_drop:<12.1f}%")

# Structured pruning
print("\n=== Structured Pruning ===\n")

structured_model = copy.deepcopy(model)
prune_info = MagnitudePruner.structured_prune(structured_model, sparsity=0.3)

for name, info in prune_info.items():
    print(f"  {name}: {info['pruned_neurons']}/{info['total_neurons']} neurons pruned "
          f"({info['kept_neurons']} kept)")

structured_acc = evaluate_model(structured_model, X_test, y_test)
structured_sparsity = compute_sparsity(structured_model)
print(f"\n  Structured pruning accuracy: {structured_acc*100:.1f}%")
print(f"  Structured pruning sparsity: {structured_sparsity:.1f}%")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Accuracy vs Sparsity
sparsities = [r[0]*100 for r in results_unstructured]
accuracies = [r[2]*100 for r in results_unstructured]

axes[0].plot(sparsities, accuracies, 'ro-', linewidth=2, markersize=8)
axes[0].axhline(y=original_acc*100, color='blue', linestyle='--', label='Original', alpha=0.7)
axes[0].set_xlabel('Target Sparsity (%)', fontsize=12)
axes[0].set_ylabel('Accuracy (%)', fontsize=12)
axes[0].set_title('Accuracy vs Sparsity', fontsize=12, weight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Weight distribution before/after pruning
original_weights = torch.cat([p.flatten() for p in model.parameters()]).detach().numpy()
pruned_50 = copy.deepcopy(model)
MagnitudePruner.unstructured_prune(pruned_50, 0.5)
pruned_weights = torch.cat([p.flatten() for p in pruned_50.parameters()]).detach().numpy()

axes[1].hist(original_weights, bins=100, alpha=0.5, label='Original', color='blue', density=True)
axes[1].hist(pruned_weights[pruned_weights != 0], bins=100, alpha=0.5, label='After Pruning (50%)', 
             color='red', density=True)
axes[1].set_xlabel('Weight Value')
axes[1].set_ylabel('Density')
axes[1].set_title('Weight Distribution', fontsize=12, weight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Sparsity pattern visualization
sample_weight = list(model.parameters())[0].data[:32, :32]
pruned_weight = list(pruned_50.parameters())[0].data[:32, :32]

axes[2].imshow((pruned_weight != 0).float().numpy(), cmap='Blues', aspect='auto')
axes[2].set_title('Sparsity Pattern (50% pruned)', fontsize=12, weight='bold')
axes[2].set_xlabel('Input Dimension')
axes[2].set_ylabel('Output Dimension')

plt.tight_layout()
plt.show()

print("\n剪枝核心要点：")
print("- 非结构化剪枝：灵活，但需要稀疏硬件支持才能加速")
print("- 结构化剪枝：直接减少计算量，但灵活性较低")
print("- 50%稀疏度通常对精度影响很小")
print("- 90%+稀疏度需要微调恢复精度")
print("- 剪枝后建议微调几个epoch恢复性能")
print("\n✓ 模型剪枝完成！")

## 5. 知识蒸馏 (Knowledge Distillation)

### 5.1 核心思想

**知识蒸馏**：用大模型（教师）的知识指导小模型（学生）的训练。

**为什么有效？**
- 教师模型的软标签（soft labels）包含类间关系信息
- 例如：猫的图片，教师输出 [猫:0.8, 狗:0.15, 兔:0.05]
- 这比硬标签 [猫:1, 狗:0, 兔:0] 包含更多信息（"暗知识"）

### 5.2 蒸馏损失

$$\mathcal{L} = \alpha \cdot \mathcal{L}_{CE}(y, \hat{y}_{student}) + (1-\alpha) \cdot T^2 \cdot \text{KL}\left(\frac{\hat{y}_{teacher}}{T} \| \frac{\hat{y}_{student}}{T}\right)$$

其中：
- $\alpha$：硬标签损失权重（通常0.1-0.5）
- $T$：温度参数（通常2-10）
- $T^2$：补偿温度对梯度的缩放

### 5.3 温度参数

**温度T的作用**：控制softmax输出的"平滑度"

$$\text{softmax}(z_i / T) = \frac{e^{z_i/T}}{\sum_j e^{z_j/T}}$$

- $T=1$：标准softmax（尖锐分布）
- $T>1$：更平滑的分布（更多暗知识）
- $T→∞$：均匀分布

### 5.4 典型应用

| 方法 | 教师 | 学生 | 压缩比 | 性能保留 |
|------|------|------|--------|---------|
| DistilBERT | BERT-base | 6层BERT | 40% | 97% |
| TinyBERT | BERT-base | 4层BERT | 56% | 96% |
| MiniLM | BERT-large | 6层BERT | 50% | 99% |

In [ ]:
# 🔬 Micro Practice: Knowledge Distillation
# Goal: Implement teacher-student knowledge transfer
# Expected outcome: Understand how to compress models via distillation

class TeacherModel(nn.Module):
    """Large teacher model"""
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(TeacherModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.net(x)

class StudentModel(nn.Module):
    """Small student model (fewer layers, smaller hidden dim)"""
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(StudentModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.net(x)

def distillation_loss(student_logits, teacher_logits, labels, temperature=3.0, alpha=0.5):
    """
    Knowledge Distillation Loss
    
    L = α * CE(student, labels) + (1-α) * KL(soft_teacher || soft_student) * T²
    
    Args:
        student_logits: Student model output
        teacher_logits: Teacher model output
        labels: Ground truth labels
        temperature: Softmax temperature (higher = softer)
        alpha: Weight for hard label loss
    """
    # Hard label loss (standard cross-entropy)
    hard_loss = nn.functional.cross_entropy(student_logits, labels)
    
    # Soft label loss (KL divergence with temperature)
    soft_student = nn.functional.log_softmax(student_logits / temperature, dim=-1)
    soft_teacher = nn.functional.softmax(teacher_logits / temperature, dim=-1)
    soft_loss = nn.functional.kl_div(soft_student, soft_teacher, reduction='batchmean')
    
    # Combined loss (T² scaling for soft loss)
    total_loss = alpha * hard_loss + (1 - alpha) * soft_loss * (temperature ** 2)
    
    return total_loss, hard_loss.item(), soft_loss.item()

# Setup
print("=== Knowledge Distillation ===\n")

input_dim, output_dim = 64, 10
teacher_hidden = 256
student_hidden = 64

# Create models
teacher = TeacherModel(input_dim, teacher_hidden, output_dim)
student_distilled = StudentModel(input_dim, student_hidden, output_dim)
student_standard = StudentModel(input_dim, student_hidden, output_dim)

# Model size comparison
teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student_distilled.parameters())

print(f"Teacher model: {teacher_params:,} parameters ({teacher_params*4/1e6:.2f} MB)")
print(f"Student model: {student_params:,} parameters ({student_params*4/1e6:.2f} MB)")
print(f"Compression:   {teacher_params/student_params:.1f}x smaller")
print()

# Generate training data
X_train = torch.randn(1000, input_dim)
y_train = torch.randint(0, output_dim, (1000,))
X_test = torch.randn(200, input_dim)
y_test = torch.randint(0, output_dim, (200,))

# Step 1: Train teacher
print("Step 1: Training teacher model...")
teacher_optimizer = torch.optim.Adam(teacher.parameters(), lr=1e-3)
teacher_losses = []

for epoch in range(100):
    teacher_optimizer.zero_grad()
    output = teacher(X_train)
    loss = nn.functional.cross_entropy(output, y_train)
    loss.backward()
    teacher_optimizer.step()
    teacher_losses.append(loss.item())

teacher.eval()
with torch.no_grad():
    teacher_acc = (teacher(X_test).argmax(dim=-1) == y_test).float().mean().item()
print(f"  Teacher accuracy: {teacher_acc*100:.1f}%")

# Step 2: Train student with distillation
print("\nStep 2: Training student with distillation...")
student_optimizer = torch.optim.Adam(student_distilled.parameters(), lr=1e-3)
distill_losses = []

for epoch in range(100):
    student_optimizer.zero_grad()
    
    student_logits = student_distilled(X_train)
    with torch.no_grad():
        teacher_logits = teacher(X_train)
    
    loss, hard_l, soft_l = distillation_loss(
        student_logits, teacher_logits, y_train, 
        temperature=3.0, alpha=0.3
    )
    loss.backward()
    student_optimizer.step()
    distill_losses.append(loss.item())

student_distilled.eval()
with torch.no_grad():
    distilled_acc = (student_distilled(X_test).argmax(dim=-1) == y_test).float().mean().item()
print(f"  Distilled student accuracy: {distilled_acc*100:.1f}%")

# Step 3: Train student without distillation (baseline)
print("\nStep 3: Training student without distillation...")
standard_optimizer = torch.optim.Adam(student_standard.parameters(), lr=1e-3)
standard_losses = []

for epoch in range(100):
    standard_optimizer.zero_grad()
    output = student_standard(X_train)
    loss = nn.functional.cross_entropy(output, y_train)
    loss.backward()
    standard_optimizer.step()
    standard_losses.append(loss.item())

student_standard.eval()
with torch.no_grad():
    standard_acc = (student_standard(X_test).argmax(dim=-1) == y_test).float().mean().item()
print(f"  Standard student accuracy: {standard_acc*100:.1f}%")

# Temperature analysis
print("\n=== Temperature Effect ===\n")
temperatures = [1.0, 2.0, 3.0, 5.0, 10.0, 20.0]

# Show softmax output at different temperatures
sample_logits = torch.tensor([2.0, 1.0, 0.5, -0.5, -1.0])
print(f"{'Temperature':<15} {'Softmax Distribution':<50}")
print("-" * 65)

for T in temperatures:
    probs = torch.softmax(sample_logits / T, dim=-1)
    prob_str = " ".join([f"{p:.3f}" for p in probs])
    entropy = -(probs * probs.log()).sum().item()
    print(f"T = {T:<10.1f} [{prob_str}]  entropy={entropy:.3f}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training loss comparison
axes[0].plot(teacher_losses, label='Teacher', linewidth=2)
axes[0].plot(distill_losses, label='Student (Distilled)', linewidth=2, linestyle='--')
axes[0].plot(standard_losses, label='Student (Standard)', linewidth=2, linestyle=':')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss', fontsize=12, weight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy comparison
models = ['Teacher', 'Student\n(Distilled)', 'Student\n(Standard)']
accuracies = [teacher_acc * 100, distilled_acc * 100, standard_acc * 100]
params = [teacher_params, student_params, student_params]
colors = ['#2196F3', '#4CAF50', '#FF9800']

axes[1].bar(models, accuracies, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Model Accuracy', fontsize=12, weight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

for i, (acc, p) in enumerate(zip(accuracies, params)):
    axes[1].text(i, acc + 0.5, f'{acc:.1f}%\n({p:,} params)', ha='center', fontsize=9)

# Temperature effect on softmax
for T in [1.0, 3.0, 10.0]:
    probs = torch.softmax(sample_logits / T, dim=-1).numpy()
    axes[2].plot(probs, 'o-', label=f'T={T}', linewidth=2, markersize=8)

axes[2].set_xlabel('Class Index')
axes[2].set_ylabel('Probability')
axes[2].set_title('Temperature Effect on Softmax', fontsize=12, weight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n知识蒸馏核心要点：")
print(f"- 教师模型: {teacher_params:,} 参数, 准确率 {teacher_acc*100:.1f}%")
print(f"- 蒸馏学生: {student_params:,} 参数, 准确率 {distilled_acc*100:.1f}%")
print(f"- 标准学生: {student_params:,} 参数, 准确率 {standard_acc*100:.1f}%")
print(f"- 压缩比: {teacher_params/student_params:.1f}x")
print("- 高温度使softmax更平滑，传递更多'暗知识'")
print("- α控制硬标签和软标签的权重平衡")
print("\n✓ 知识蒸馏完成！")

## 6. 模型导出与ONNX优化

### 6.1 ONNX简介

**ONNX (Open Neural Network Exchange)** 是一个开放的模型交换格式：

- **跨框架**：PyTorch → ONNX → TensorRT/OpenVINO/CoreML
- **图优化**：常量折叠、算子融合、死代码消除
- **硬件加速**：利用特定硬件的优化kernel

### 6.2 导出流程

```
PyTorch Model
    ↓ torch.onnx.export()
ONNX Model (.onnx)
    ↓ onnx.optimizer
Optimized ONNX Model
    ↓ onnxruntime / TensorRT
Hardware-Optimized Inference
```

### 6.3 图优化技术

| 优化 | 说明 | 效果 |
|------|------|------|
| **常量折叠** | 预计算常量表达式 | 减少运行时计算 |
| **算子融合** | 合并连续操作（如Linear+ReLU） | 减少kernel启动开销 |
| **死代码消除** | 移除未使用的计算节点 | 减少无效计算 |
| **形状推断** | 静态传播张量形状 | 启用更多优化 |

### 6.4 PyTorch导出示例

```python
# 导出到ONNX
torch.onnx.export(
    model,
    dummy_input,
    "model.onnx",
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}},  # 动态batch
    opset_version=17
)

# 使用ONNX Runtime推理
import onnxruntime as ort
session = ort.InferenceSession("model.onnx")
output = session.run(None, {'input': input_data})
```

In [ ]:
# 🔬 Micro Practice: Model Export and ONNX Optimization
# Goal: Understand model export pipeline and graph optimization
# Expected outcome: Know how to export and optimize models for deployment

class ONNXExportSimulator:
    """
    Simulate ONNX export and optimization pipeline
    In practice, use torch.onnx.export() and onnxruntime
    """
    
    @staticmethod
    def analyze_model_graph(model, input_shape):
        """Analyze model computation graph"""
        # Count operations
        ops = {'linear': 0, 'activation': 0, 'normalization': 0, 'other': 0}
        
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                ops['linear'] += 1
            elif isinstance(module, (nn.ReLU, nn.GELU, nn.Sigmoid, nn.Tanh)):
                ops['activation'] += 1
            elif isinstance(module, (nn.LayerNorm, nn.BatchNorm1d)):
                ops['normalization'] += 1
        
        return ops
    
    @staticmethod
    def simulate_graph_optimizations(model, input_shape):
        """
        Simulate ONNX graph optimizations:
        1. Constant folding: pre-compute constant expressions
        2. Operator fusion: combine sequential ops into one
        3. Dead code elimination: remove unused nodes
        4. Shape inference: propagate shapes through graph
        """
        optimizations = []
        
        # 1. Constant folding
        n_constants = sum(1 for p in model.parameters() if not p.requires_grad)
        optimizations.append({
            'name': 'Constant Folding',
            'description': 'Pre-compute constant expressions',
            'nodes_removed': n_constants,
            'speedup': 1.05,
        })
        
        # 2. Operator fusion (e.g., Linear + ReLU → FusedLinearReLU)
        fusable_pairs = 0
        modules = list(model.modules())
        for i in range(len(modules) - 1):
            if isinstance(modules[i], nn.Linear) and isinstance(modules[i+1], nn.ReLU):
                fusable_pairs += 1
        
        optimizations.append({
            'name': 'Operator Fusion',
            'description': f'Fuse {fusable_pairs} Linear+ReLU pairs',
            'nodes_removed': fusable_pairs,
            'speedup': 1.0 + fusable_pairs * 0.05,
        })
        
        # 3. Dead code elimination
        optimizations.append({
            'name': 'Dead Code Elimination',
            'description': 'Remove unused computation nodes',
            'nodes_removed': 0,
            'speedup': 1.0,
        })
        
        # 4. Shape inference
        optimizations.append({
            'name': 'Shape Inference',
            'description': 'Static shape propagation for optimization',
            'nodes_removed': 0,
            'speedup': 1.02,
        })
        
        total_speedup = 1.0
        for opt in optimizations:
            total_speedup *= opt['speedup']
        
        return optimizations, total_speedup
    
    @staticmethod
    def simulate_export(model, input_shape):
        """Simulate ONNX export process"""
        x = torch.randn(*input_shape)
        
        # Trace the model
        model.eval()
        with torch.no_grad():
            output = model(x)
        
        # Collect export info
        param_count = sum(p.numel() for p in model.parameters())
        model_size = sum(p.numel() * p.element_size() for p in model.parameters())
        
        return {
            'input_shape': input_shape,
            'output_shape': tuple(output.shape),
            'param_count': param_count,
            'model_size_mb': model_size / 1e6,
            'format': 'ONNX (Open Neural Network Exchange)',
            'opset_version': 17,
        }

# Demo ONNX export and optimization
print("=== Model Export & ONNX Optimization ===\n")

# Create a model
model = nn.Sequential(
    nn.Linear(256, 512),
    nn.ReLU(),
    nn.LayerNorm(512),
    nn.Linear(512, 512),
    nn.ReLU(),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

simulator = ONNXExportSimulator()
input_shape = (1, 256)

# Step 1: Analyze graph
print("Step 1: Model Graph Analysis")
ops = simulator.analyze_model_graph(model, input_shape)
print(f"  Linear layers:       {ops['linear']}")
print(f"  Activation layers:   {ops['activation']}")
print(f"  Normalization layers: {ops['normalization']}")
print(f"  Total operations:    {sum(ops.values())}")

# Step 2: Export
print("\nStep 2: ONNX Export")
export_info = simulator.simulate_export(model, input_shape)
print(f"  Input shape:  {export_info['input_shape']}")
print(f"  Output shape: {export_info['output_shape']}")
print(f"  Parameters:   {export_info['param_count']:,}")
print(f"  Model size:   {export_info['model_size_mb']:.2f} MB")
print(f"  ONNX opset:   {export_info['opset_version']}")

# Step 3: Graph optimizations
print("\nStep 3: Graph Optimizations")
optimizations, total_speedup = simulator.simulate_graph_optimizations(model, input_shape)

for opt in optimizations:
    status = "✓" if opt['speedup'] > 1.0 else "○"
    print(f"  {status} {opt['name']}: {opt['description']} ({opt['speedup']:.2f}x)")

print(f"\n  Total optimization speedup: {total_speedup:.2f}x")

# Step 4: Benchmark
print("\nStep 4: Benchmark")
model.eval()
x = torch.randn(*input_shape)

# Original PyTorch
times_pytorch = []
with torch.no_grad():
    for _ in range(200):
        start = time.time()
        _ = model(x)
        times_pytorch.append(time.time() - start)

pytorch_latency = np.mean(times_pytorch) * 1000
onnx_latency = pytorch_latency / total_speedup

print(f"  PyTorch inference:  {pytorch_latency:.3f} ms")
print(f"  ONNX optimized:    {onnx_latency:.3f} ms (simulated)")
print(f"  Speedup:           {total_speedup:.2f}x")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Export pipeline
stages = ['PyTorch\nModel', 'ONNX\nExport', 'Graph\nOptimization', 'Hardware\nAcceleration']
stage_times = [pytorch_latency, pytorch_latency * 0.95, onnx_latency, onnx_latency * 0.6]
colors = ['#F44336', '#FF9800', '#4CAF50', '#2196F3']

axes[0].bar(stages, stage_times, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Optimization Pipeline Stages', fontsize=12, weight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Optimization breakdown
opt_names = [o['name'] for o in optimizations]
opt_speedups = [o['speedup'] for o in optimizations]

axes[1].barh(opt_names, opt_speedups, color=['#4CAF50', '#2196F3', '#FF9800', '#9C27B0'], 
             alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Speedup (x)')
axes[1].set_title('Graph Optimization Breakdown', fontsize=12, weight='bold')
axes[1].axvline(x=1.0, color='gray', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\nONNX导出最佳实践：")
print("1. 使用torch.onnx.export()导出模型")
print("2. 指定dynamic_axes支持动态batch size")
print("3. 使用onnxruntime进行推理")
print("4. 启用图优化（OrtSession with optimization_level）")
print("5. 结合量化进一步加速")
print("\n✓ ONNX导出与优化完成！")

## 7. 硬件加速 (Hardware Acceleration)

### 7.1 加速后端概览

不同硬件平台有专门的推理加速框架：

| 框架 | 目标硬件 | 关键特性 |
|------|---------|---------|
| **TensorRT** | NVIDIA GPU | 层融合、精度校准、kernel自动调优 |
| **OpenVINO** | Intel CPU/GPU | 模型优化器、推理引擎、INT8量化 |
| **CoreML** | Apple芯片 | Neural Engine、GPU计算、ANE加速 |
| **TFLite** | ARM CPU/GPU | 移动端优化、代理机制、NNAPI |
| **ONNX Runtime** | 跨平台 | 图优化、多后端支持、动态量化 |

### 7.2 TensorRT优化流程

```
PyTorch Model → ONNX → TensorRT Engine
                         ├── 层融合（Conv+BN+ReLU → 1个kernel）
                         ├── 精度校准（FP32→FP16/INT8）
                         ├── Kernel自动调优（选择最快实现）
                         └── 动态内存管理
```

### 7.3 选择加速后端

**决策因素**：
1. **目标硬件**：GPU → TensorRT，CPU → OpenVINO，移动端 → TFLite
2. **延迟要求**：实时 → GPU加速，批处理 → CPU也可
3. **精度要求**：FP16通常无损，INT8需要校准
4. **部署环境**：云端 → TensorRT，边缘 → OpenVINO/TFLite

In [ ]:
# 🔬 Micro Practice: Hardware Acceleration Concepts
# Goal: Understand different hardware acceleration options
# Expected outcome: Know how to choose the right acceleration backend

class HardwareAccelerationSimulator:
    """
    Simulate different hardware acceleration backends
    In practice, each backend has its own SDK and optimization pipeline
    """
    
    @staticmethod
    def simulate_tensorrt(model, input_shape, precision='fp16'):
        """
        Simulate TensorRT optimization
        TensorRT: NVIDIA's inference optimizer for GPU
        - Layer fusion
        - Kernel auto-tuning
        - Precision calibration (FP16/INT8)
        - Dynamic tensor memory
        """
        x = torch.randn(*input_shape)
        
        # Simulate TensorRT optimizations
        optimizations = {
            'layer_fusion': True,
            'kernel_autotuning': True,
            'precision': precision,
            'dynamic_memory': True,
        }
        
        # Benchmark original
        model.eval()
        times_original = []
        with torch.no_grad():
            for _ in range(100):
                start = time.time()
                _ = model(x)
                times_original.append(time.time() - start)
        
        # Simulated speedup factors
        speedup_factors = {
            'fp32': 1.5,   # Graph optimization only
            'fp16': 2.5,   # + half precision
            'int8': 4.0,   # + quantization
        }
        
        speedup = speedup_factors.get(precision, 1.0)
        simulated_latency = np.mean(times_original) * 1000 / speedup
        
        return {
            'backend': 'TensorRT',
            'precision': precision,
            'original_latency_ms': np.mean(times_original) * 1000,
            'optimized_latency_ms': simulated_latency,
            'speedup': speedup,
            'optimizations': optimizations,
        }
    
    @staticmethod
    def simulate_openvino(model, input_shape):
        """
        Simulate OpenVINO optimization
        OpenVINO: Intel's inference optimizer for CPU
        - Model optimizer (graph transformations)
        - Inference engine (hardware-specific kernels)
        - INT8 quantization with accuracy checker
        """
        x = torch.randn(*input_shape)
        
        model.eval()
        times = []
        with torch.no_grad():
            for _ in range(100):
                start = time.time()
                _ = model(x)
                times.append(time.time() - start)
        
        return {
            'backend': 'OpenVINO',
            'precision': 'INT8',
            'original_latency_ms': np.mean(times) * 1000,
            'optimized_latency_ms': np.mean(times) * 1000 / 3.0,
            'speedup': 3.0,
            'target': 'Intel CPU',
        }
    
    @staticmethod
    def simulate_coreml(model, input_shape):
        """
        Simulate CoreML optimization
        CoreML: Apple's ML framework
        - Neural Engine acceleration
        - GPU compute
        - CPU fallback
        """
        x = torch.randn(*input_shape)
        
        model.eval()
        times = []
        with torch.no_grad():
            for _ in range(100):
                start = time.time()
                _ = model(x)
                times.append(time.time() - start)
        
        return {
            'backend': 'CoreML',
            'precision': 'FP16',
            'original_latency_ms': np.mean(times) * 1000,
            'optimized_latency_ms': np.mean(times) * 1000 / 2.0,
            'speedup': 2.0,
            'target': 'Apple Neural Engine',
        }

# Compare hardware acceleration backends
print("=== Hardware Acceleration Comparison ===\n")

model = nn.Sequential(
    nn.Linear(256, 512), nn.ReLU(),
    nn.Linear(512, 512), nn.ReLU(),
    nn.Linear(512, 256), nn.ReLU(),
    nn.Linear(256, 10)
)

input_shape = (1, 256)
simulator = HardwareAccelerationSimulator()

# Simulate different backends
results = []

# TensorRT with different precisions
for precision in ['fp32', 'fp16', 'int8']:
    result = simulator.simulate_tensorrt(model, input_shape, precision)
    results.append(result)
    print(f"TensorRT ({precision}): {result['original_latency_ms']:.3f}ms → "
          f"{result['optimized_latency_ms']:.3f}ms ({result['speedup']:.1f}x)")

# OpenVINO
result = simulator.simulate_openvino(model, input_shape)
results.append(result)
print(f"OpenVINO (INT8):  {result['original_latency_ms']:.3f}ms → "
      f"{result['optimized_latency_ms']:.3f}ms ({result['speedup']:.1f}x)")

# CoreML
result = simulator.simulate_coreml(model, input_shape)
results.append(result)
print(f"CoreML (FP16):    {result['original_latency_ms']:.3f}ms → "
      f"{result['optimized_latency_ms']:.3f}ms ({result['speedup']:.1f}x)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Speedup comparison
backends = [f"{r['backend']}\n({r['precision']})" for r in results]
speedups = [r['speedup'] for r in results]
colors = ['#FF9800', '#F44336', '#9C27B0', '#2196F3', '#4CAF50']

axes[0].barh(backends, speedups, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Speedup (x)', fontsize=12)
axes[0].set_title('Hardware Acceleration Speedup', fontsize=12, weight='bold')
axes[0].grid(True, alpha=0.3, axis='x')
axes[0].axvline(x=1, color='gray', linestyle='--', alpha=0.5)

# Latency comparison
original = [r['original_latency_ms'] for r in results]
optimized = [r['optimized_latency_ms'] for r in results]

x_pos = np.arange(len(backends))
width = 0.35

axes[1].bar(x_pos - width/2, original, width, label='Original', color='red', alpha=0.5)
axes[1].bar(x_pos + width/2, optimized, width, label='Optimized', color='green', alpha=0.7)
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Latency: Original vs Optimized', fontsize=12, weight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([r['backend'] + '\n' + r['precision'] for r in results], fontsize=8)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Hardware selection guide
print("\n硬件加速选择指南：")
print("┌─────────────┬──────────────┬──────────────┬──────────────┐")
print("│ 后端        │ 目标硬件     │ 最佳场景     │ 典型加速     │")
print("├─────────────┼──────────────┼──────────────┼──────────────┤")
print("│ TensorRT    │ NVIDIA GPU   │ 数据中心推理 │ 2-5x         │")
print("│ OpenVINO    │ Intel CPU    │ 边缘计算     │ 2-4x         │")
print("│ CoreML      │ Apple芯片    │ iOS/macOS    │ 2-3x         │")
print("│ TFLite      │ ARM CPU/GPU  │ 移动端       │ 2-4x         │")
print("│ ONNX Runtime│ 跨平台      │ 通用部署     │ 1.5-3x       │")
print("└─────────────┴──────────────┴──────────────┴──────────────┘")
print("\n✓ 硬件加速对比完成！")

## 8. 综合优化与总结

### 8.1 优化决策树

**根据约束选择优化技术**：

```
推理优化需求
├── 延迟敏感（实时应用）
│   ├── GPU可用 → TensorRT + FP16/INT8
│   ├── CPU部署 → ONNX + OpenVINO + INT8
│   └── 移动端 → TFLite + INT8 + 剪枝
├── 吞吐量优先（批处理）
│   ├── 动态批处理 + 量化
│   └── 模型并行 + 流水线
├── 内存受限
│   ├── 量化（4x压缩）
│   ├── 剪枝（2-5x压缩）
│   └── 蒸馏（更小模型）
└── 精度敏感
    ├── 静态量化 + 校准
    ├── QAT（量化感知训练）
    └── 蒸馏（保持性能）
```

**优化技术对比**：

| 技术 | 压缩比 | 精度损失 | 实现难度 | 需要重训练 |
|------|--------|---------|---------|-----------|
| 动态量化 | 2-4x | <1% | 简单 | 否 |
| 静态量化 | 2-4x | <0.5% | 中等 | 否（需校准） |
| 非结构化剪枝 | 2-10x | 1-3% | 中等 | 是 |
| 结构化剪枝 | 2-5x | 1-5% | 较难 | 是 |
| 知识蒸馏 | 2-10x | 1-5% | 较难 | 是 |
| ONNX优化 | 1.2-2x | 0% | 简单 | 否 |
| TensorRT | 2-5x | <1% | 中等 | 否 |

In [ ]:
# 🎯 Comprehensive Example: Optimization Pipeline
# Goal: Combine all optimization techniques
# Expected outcome: Understand end-to-end inference optimization

class InferenceOptimizationPipeline:
    """
    Complete inference optimization pipeline combining multiple techniques
    """
    
    def __init__(self, model, input_shape):
        self.original_model = model
        self.input_shape = input_shape
        self.results = {}
    
    def benchmark(self, model, name, n_runs=100):
        """Benchmark a model's inference performance"""
        model.eval()
        x = torch.randn(*self.input_shape)
        
        # Warmup
        with torch.no_grad():
            for _ in range(10):
                _ = model(x)
        
        # Benchmark
        times = []
        with torch.no_grad():
            for _ in range(n_runs):
                start = time.time()
                output = model(x)
                times.append(time.time() - start)
        
        # Model size
        param_count = sum(p.numel() for p in model.parameters())
        param_memory = sum(p.numel() * p.element_size() for p in model.parameters())
        
        # Sparsity
        total_zeros = sum((p == 0).sum().item() for p in model.parameters())
        total_params = sum(p.numel() for p in model.parameters())
        sparsity = total_zeros / total_params * 100
        
        result = {
            'name': name,
            'mean_latency': np.mean(times) * 1000,  # ms
            'p95_latency': np.percentile(times, 95) * 1000,
            'throughput': 1.0 / np.mean(times),
            'param_count': param_count,
            'memory_mb': param_memory / 1e6,
            'sparsity': sparsity,
        }
        
        self.results[name] = result
        return result
    
    def apply_quantization(self, model, bits=8):
        """Apply simulated quantization"""
        import copy
        q_model = copy.deepcopy(model)
        
        with torch.no_grad():
            for param in q_model.parameters():
                # Simulate quantization
                scale = (param.max() - param.min()) / (2**bits - 1)
                if scale > 0:
                    param.data = torch.round(param.data / scale) * scale
        
        return q_model
    
    def apply_pruning(self, model, sparsity=0.5):
        """Apply magnitude pruning"""
        import copy
        p_model = copy.deepcopy(model)
        
        with torch.no_grad():
            for param in p_model.parameters():
                if param.dim() >= 2:
                    threshold = torch.quantile(param.abs().float(), sparsity)
                    mask = param.abs() > threshold
                    param.data *= mask.float()
        
        return p_model
    
    def run_pipeline(self):
        """Run complete optimization pipeline"""
        print("=== Inference Optimization Pipeline ===\n")
        
        # 1. Baseline
        print("Step 1: Baseline benchmark...")
        self.benchmark(self.original_model, "Baseline (FP32)")
        
        # 2. Quantization
        print("Step 2: Applying INT8 quantization...")
        q_model = self.apply_quantization(self.original_model, bits=8)
        self.benchmark(q_model, "Quantized (INT8)")
        
        # 3. Pruning
        print("Step 3: Applying 50% pruning...")
        p_model = self.apply_pruning(self.original_model, sparsity=0.5)
        self.benchmark(p_model, "Pruned (50%)")
        
        # 4. Pruning + Quantization
        print("Step 4: Pruning + Quantization...")
        pq_model = self.apply_quantization(p_model, bits=8)
        self.benchmark(pq_model, "Pruned + Quantized")
        
        # 5. Aggressive pruning
        print("Step 5: Aggressive pruning (80%)...")
        ap_model = self.apply_pruning(self.original_model, sparsity=0.8)
        self.benchmark(ap_model, "Pruned (80%)")
        
        print("\nPipeline complete!\n")
        return self.results
    
    def report(self):
        """Generate optimization report"""
        baseline = self.results.get("Baseline (FP32)")
        if not baseline:
            print("Run pipeline first!")
            return
        
        print(f"{'Method':<25} {'Latency(ms)':<14} {'Memory(MB)':<12} {'Sparsity':<10} {'Speedup':<10}")
        print("-" * 71)
        
        for name, r in self.results.items():
            speedup = baseline['mean_latency'] / r['mean_latency']
            print(f"{name:<25} {r['mean_latency']:<14.3f} {r['memory_mb']:<12.2f} "
                  f"{r['sparsity']:<10.1f}% {speedup:<10.2f}x")

# Build and run pipeline
model = nn.Sequential(
    nn.Linear(512, 1024),
    nn.ReLU(),
    nn.Linear(1024, 1024),
    nn.ReLU(),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Linear(512, 10)
)

pipeline = InferenceOptimizationPipeline(model, input_shape=(1, 512))
results = pipeline.run_pipeline()
pipeline.report()

# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

names = list(results.keys())
latencies = [results[n]['mean_latency'] for n in names]
memories = [results[n]['memory_mb'] for n in names]
sparsities = [results[n]['sparsity'] for n in names]

colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']

# Latency comparison
axes[0].barh(names, latencies, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Latency (ms)')
axes[0].set_title('Inference Latency', fontsize=12, weight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# Memory comparison
axes[1].barh(names, memories, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Memory (MB)')
axes[1].set_title('Model Memory', fontsize=12, weight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

# Sparsity
axes[2].barh(names, sparsities, color=colors, alpha=0.7, edgecolor='black')
axes[2].set_xlabel('Sparsity (%)')
axes[2].set_title('Weight Sparsity', fontsize=12, weight='bold')
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n综合优化效果：")
print("- 量化：减少内存，保持精度")
print("- 剪枝：增加稀疏性，减少有效计算")
print("- 组合使用：效果叠加，可达到5-10x压缩")
print("- 实际加速需要硬件支持（稀疏计算、INT8推理）")
print("\n✓ 综合优化流水线完成！")

### 8.2 常见问题与调试

#### Q1: 应该先用哪种优化技术？

**A:** 按投入产出比排序：
1. 量化（最简单，效果显著）
2. 模型导出+图优化（ONNX）
3. 剪枝（需要微调恢复精度）
4. 知识蒸馏（需要重新训练）
5. 硬件加速（需要特定硬件）

#### Q2: 量化会损失多少精度？

**A:**
- INT8动态量化：通常损失<1%
- INT8静态量化：损失<0.5%（需要校准）
- INT4量化：损失2-5%，需要QAT
- 对于分类任务影响较小，生成任务影响较大

#### Q3: 剪枝和量化可以同时使用吗？

**A:**
- 可以，但需要注意顺序
- 推荐：先剪枝→微调→再量化
- 两者结合可以达到10x以上的压缩比
- 注意监控精度下降

#### Q4: 知识蒸馏需要多少训练数据？

**A:**
- 通常需要原始训练数据的10-50%
- 无标签数据也可以用（教师提供软标签）
- 数据质量比数量更重要
- 领域内数据效果最好

#### Q5: ONNX导出失败怎么办？

**A:**
- 检查不支持的操作（自定义算子）
- 使用`torch.jit.script`替代`torch.jit.trace`
- 简化动态控制流
- 查看ONNX算子支持列表

### 8.3 本章总结

#### 核心要点

1. **推理基础**
   - 推理优化目标：延迟、吞吐量、内存、成本
   - 先分析瓶颈，再针对性优化
   - 优化层次：算法 > 模型 > 系统 > 硬件

2. **量化**
   - 降低数值精度（FP32→INT8/INT4）
   - 动态量化：简单，适合线性层
   - 静态量化：需要校准，效果更好
   - 通常可减少2-4x内存，加速2-3x

3. **剪枝**
   - 移除不重要的权重/神经元
   - 非结构化剪枝：灵活但需要稀疏硬件支持
   - 结构化剪枝：直接减少计算量
   - 可移除50-90%的参数

4. **知识蒸馏**
   - 大模型（教师）指导小模型（学生）
   - 软标签包含更多信息
   - 可保留90-97%的性能，减少40-60%参数

5. **模型导出（ONNX）**
   - 跨框架标准格式
   - 图优化：常量折叠、算子融合
   - 部署到不同硬件平台

6. **硬件加速**
   - TensorRT（NVIDIA GPU）
   - OpenVINO（Intel CPU）
   - CoreML（Apple设备）
   - 选择取决于目标硬件

#### 优化决策流程

```
1. 分析推理瓶颈（延迟？内存？吞吐量？）
2. 尝试量化（最简单，效果显著）
3. 考虑剪枝（如果模型过大）
4. 考虑蒸馏（如果需要更小模型）
5. 导出ONNX + 硬件加速
6. 组合多种技术
```

### 8.4 下一章预告

**Module 7.2: 模型服务与部署**
- REST API服务
- 批处理推理
- 模型版本管理
- 生产环境监控

### 💡 思考题

1. 量化到INT4和INT8的精度损失差异有多大？什么场景下可以接受INT4？

2. 剪枝后的稀疏模型如何在实际硬件上获得加速？

3. 知识蒸馏中，温度参数T如何影响蒸馏效果？

4. ONNX导出时，哪些PyTorch操作可能不被支持？如何处理？

5. 如何在延迟、精度和成本之间找到最佳平衡点？

## 思考题参考答案

### 问题 1：量化会导致精度损失，如何在精度和速度之间做权衡？

量化的精度损失本质是**信息压缩**导致的量化误差。理解并控制这种误差是权衡的核心。

**量化误差的来源：**

对于对称量化，权重 $w$ 量化为 $w_q$：

$$w_q = \text{round}\left(\frac{w}{S}\right) \cdot S, \quad S = \frac{\max(|w|)}{2^{b-1} - 1}$$

误差为 $\epsilon = w - w_q$，最大误差为 $\pm \frac{S}{2}$，精度越低（b 越小）误差越大。

**不同量化精度的精度损失对比：**

| 精度格式 | 位宽 | 推理加速 | 典型精度损失 | 适用场景 |
|---------|------|---------|------------|---------|
| FP32 | 32位 | 1× 基线 | 0% | 训练、精度敏感场景 |
| FP16/BF16 | 16位 | 1.5-2× | < 0.5% | 大多数生产场景 |
| INT8 | 8位 | 2-4× | 0.5-1% | 推理优先，可接受微小精度损失 |
| INT4 | 4位 | 4-8× | 1-3% | 边缘设备，资源极度受限 |

**系统性权衡方法：**

**1. 逐层分析敏感度（Layer-wise Sensitivity Analysis）**

不同层对量化的敏感度差异很大：
- **Attention 层**：敏感度高，建议保留 FP16
- **FFN 层**：敏感度中等，INT8 通常可接受
- **Embedding 层**：词嵌入量化收益大，损失小

**混合精度量化策略（推荐）：**
```python
# 敏感层用高精度，非敏感层用低精度
quantization_config = {
    "attention.query": "fp16",   # 敏感，保留高精度
    "attention.key": "fp16",
    "ffn.linear1": "int8",       # 不敏感，量化加速
    "ffn.linear2": "int8",
    "output.classifier": "fp16"  # 输出层保留精度
}
```

**2. 量化感知训练（QAT）vs 训练后量化（PTQ）**

| 方法 | 精度损失 | 计算成本 | 适用场景 |
|------|---------|---------|---------|
| 动态量化（PTQ） | 较高（约 1-3%） | 零成本 | 快速验证 |
| 静态量化（PTQ） | 中等（约 0.5-1%） | 需校准数据集 | 平衡方案 |
| 量化感知训练（QAT） | 最低（约 0.1-0.5%） | 需重新训练 | 精度要求高 |

**决策流程：**
```
目标精度损失 < 0.5%？
    ├── 是 → 先尝试 INT8 静态量化 → 不满足 → 使用 QAT
    └── 否 → INT8 动态量化满足需求？→ 是 → 完成
                                    → 否 → 考虑 FP16
```

---

### 问题 2：模型剪枝后如何恢复精度？常见的微调策略有哪些？

剪枝后精度恢复的核心是**让残余权重重新适应新的稀疏结构**，主要通过微调（Fine-tuning）实现。

**精度下降的原因：**

剪枝将部分权重置为 0，破坏了网络原有的函数映射：
$$f_{\text{pruned}}(x) \approx f_{\text{original}}(x) - \Delta f$$

其中 $\Delta f$ 是因剪枝丢失的表达能力，需要通过微调来补偿。

**微调策略对比：**

**1. 全量微调（Full Fine-tuning）**

```python
# 剪枝后解冻所有权重重新训练
optimizer = Adam(model.parameters(), lr=1e-5)  # 小学习率
for epoch in range(3):
    train_one_epoch(model, train_loader, optimizer)
```
- 优点：精度恢复效果最好
- 缺点：计算成本高（接近重新训练）
- 适用：剪枝率 > 70%，精度损失 > 2%

**2. 渐进式剪枝 + 微调（推荐）**

不一次剪枝到目标稀疏度，而是分多轮迭代：
```
初始模型 → 剪枝 20% → 微调 1 epoch → 剪枝 20% → 微调 → ... → 目标稀疏度
```
- 每次剪枝幅度小，微调可快速恢复
- 总体精度损失远低于一次性剪枝

**3. 知识蒸馏辅助恢复（Distillation-Aided Pruning）**

将原始密集模型作为教师，剪枝后稀疏模型作为学生：

$$\mathcal{L}_{\text{total}} = \alpha \cdot \mathcal{L}_{\text{CE}} + (1-\alpha) \cdot \mathcal{L}_{\text{KD}}$$

$$\mathcal{L}_{\text{KD}} = \text{KL}(p_{\text{teacher}}(y|x, T) \| p_{\text{pruned}}(y|x, T))$$

- 教师提供软标签，帮助稀疏模型学习更好的决策边界
- 精度恢复效果优于单纯微调

**4. 结构化剪枝 + LoRA 微调**

对于大语言模型，结合参数高效微调（PEFT）：
```python
from peft import LoraConfig, get_peft_model
# 剪枝后只微调少量参数
lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"])
model = get_peft_model(pruned_model, lora_config)
```

**实践建议：** 对于目标稀疏度 < 50% 的场景，推荐使用渐进式剪枝 + 小学习率微调（3-5 个 epoch），可将精度损失控制在 1% 以内。

---

### 问题 3：知识蒸馏中温度参数 T 如何影响学习效果？

温度参数 $T$ 是知识蒸馏的核心超参数，控制**软标签（Soft Labels）的平滑程度**。

**温度缩放的数学定义：**

教师模型的软概率分布：
$$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

当 $T = 1$ 时为标准 Softmax；当 $T > 1$ 时，分布更平滑（熵增大）；当 $T \to \infty$ 时，趋近均匀分布。

**温度对学习效果的影响：**

| 温度 $T$ | 软标签特性 | 学生学到的信息 | 适用场景 |
|---------|-----------|--------------|---------|
| $T = 1$ | 尖锐分布（接近硬标签） | 仅学习预测类别 | 不推荐用于蒸馏 |
| $T = 2-4$ | 适度平滑 | 学习类别间相似关系 | 大多数分类任务 |
| $T = 5-10$ | 高度平滑 | 学习丰富的暗知识（Dark Knowledge） | 类别间关系复杂 |
| $T > 20$ | 近似均匀 | 信息噪声过大 | 不推荐 |

**"暗知识"（Dark Knowledge）的含义：**

以猫/狗/汽车三分类为例，教师对某张猫图片的输出：
```
T=1（硬标签）:  猫=0.97, 狗=0.02, 汽车=0.01
T=4（软标签）:  猫=0.60, 狗=0.30, 汽车=0.10
```

软标签揭示了"猫更像狗，而不像汽车"这一结构信息，使学生模型学到更好的特征表示。

**蒸馏损失中的温度应用：**

$$\mathcal{L}_{\text{KD}} = T^2 \cdot \text{KL}(p^T_{\text{teacher}} \| p^T_{\text{student}})$$

注意损失前乘以 $T^2$，用于补偿软标签梯度量级变化（温度升高会使梯度缩小 $T^2$ 倍）。

**超参数调优建议：**

- **初始值**：$T = 4$，这是大多数论文报告的经验最优值
- **搜索范围**：$T \in \{2, 4, 6, 8, 10\}$，在验证集上网格搜索
- **与 $\alpha$ 联合调优**：$\alpha$（蒸馏损失权重）和 $T$ 共同决定蒸馏效果，通常 $\alpha = 0.5, T = 4$ 是好的起点

---

### 问题 4：ONNX 导出后如何确保模型行为与原始 PyTorch 模型一致？

ONNX 导出过程中可能出现**算子不支持、数值精度偏差、动态形状处理错误**等问题。确保一致性需要系统性验证。

**常见不一致来源：**

| 问题类型 | 具体表现 | 解决方法 |
|---------|---------|---------|
| 算子不支持 | 某些自定义层无法导出 | 注册自定义算子或等效替换 |
| 浮点精度差异 | 输出数值有微小偏差（< 1e-5） | 设置合理容差阈值 |
| 动态形状错误 | 不同 batch size 时输出错误 | 设置 `dynamic_axes` 参数 |
| 控制流问题 | `if/for` 语句导致图分叉 | 使用 `torch.onnx.export` 时设置 `opset_version` |

**系统性验证流程：**

**1. 数值一致性验证（最重要）**

```python
import torch
import onnxruntime as ort
import numpy as np

def verify_onnx_consistency(pytorch_model, onnx_path, test_inputs, rtol=1e-3, atol=1e-5):
    # PyTorch 推理
    pytorch_model.eval()
    with torch.no_grad():
        pt_output = pytorch_model(**test_inputs).numpy()
    
    # ONNX Runtime 推理
    sess = ort.InferenceSession(onnx_path)
    onnx_inputs = {k: v.numpy() for k, v in test_inputs.items()}
    onnx_output = sess.run(None, onnx_inputs)[0]
    
    # 对比
    np.testing.assert_allclose(pt_output, onnx_output, rtol=rtol, atol=atol)
    print(f"最大绝对误差: {np.max(np.abs(pt_output - onnx_output)):.2e}")
    print("✓ ONNX 输出与 PyTorch 一致")
```

**2. 多样化测试用例**

```python
test_cases = [
    {"batch_size": 1, "seq_len": 16},   # 最小输入
    {"batch_size": 8, "seq_len": 128},  # 典型输入
    {"batch_size": 32, "seq_len": 512}, # 最大输入
    # 边界条件：全零输入、极大/极小值
]
```

**3. 模型图验证**

```python
import onnx
model = onnx.load(onnx_path)
onnx.checker.check_model(model)  # 图结构合法性检查

# 可视化（使用 Netron 工具）
# 对比 PyTorch 和 ONNX 的算子数量和类型
```

**4. 性能对比**

```python
# 确保 ONNX 确实比 PyTorch 快
onnx_throughput = benchmark(onnx_session, test_inputs)
pytorch_throughput = benchmark(pytorch_model, test_inputs)
speedup = onnx_throughput / pytorch_throughput
assert speedup > 1.0, f"ONNX 未加速，可能优化未生效 (speedup={speedup:.2f}x)"
```

---

### 问题 5：如何为特定硬件选择最合适的推理优化方案？

硬件选择决定了可用的优化工具链，需要**理解硬件特性 → 分析模型特征 → 匹配最优方案**。

**主要硬件平台与推荐方案：**

| 硬件平台 | 推荐框架 | 最优精度格式 | 典型加速比 |
|---------|---------|------------|----------|
| NVIDIA GPU（数据中心） | TensorRT | INT8/FP16 | 3-8× |
| NVIDIA GPU（消费级） | TorchScript + CUDA | FP16 | 1.5-3× |
| Intel CPU | OpenVINO | INT8 | 2-4× |
| ARM CPU（手机/边缘） | TFLite / CoreML | INT8 | 2-5× |
| Apple Silicon（M系列） | CoreML / MPS | FP16 | 2-4× |
| 专用 NPU（华为、高通） | 厂商 SDK | INT8/INT4 | 5-20× |

**决策树：**

```
确定部署目标硬件
    ↓
NVIDIA GPU？
    ├── 是（数据中心 A100/H100）→ TensorRT + INT8 量化
    ├── 是（消费级 RTX）→ TorchScript + FP16 + CUDA
    └── 否
         ↓
        Intel CPU？
            ├── 是 → OpenVINO + INT8 量化
            └── 否
                 ↓
                移动端 ARM？
                    ├── 是（Android）→ TFLite + INT8
                    ├── 是（iOS）→ CoreML + FP16/INT8
                    └── 否 → ONNX Runtime（通用方案）
```

**Roofline 模型指导优化方向：**

通过计算模型的**算术强度**来判断瓶颈：

$$\text{算术强度} = \frac{\text{FLOPs}}{\text{内存访问量（Bytes）}}$$

- **算术强度 < 硬件 Ridge Point**：内存带宽瓶颈 → 优先量化（减少内存传输）
- **算术强度 > Ridge Point**：计算瓶颈 → 优先使用张量核心（Tensor Core）/ 算子融合

**实践建议：**

1. 先做 Profiling，识别真正的瓶颈（不要盲目优化）
2. 按"收益/代价"排序：量化 > ONNX 导出 > 算子融合 > 硬件专用加速
3. 在目标硬件上做端到端基准测试，而非依赖理论加速比